In [26]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_samples
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import os
os.getcwd()

'c:\\Users\\Bruno Abello\\Desktop\\Personal-projects\\Proyectos de Astronomía\\Laboratorio 4'

In [27]:
df = pd.read_csv("dataset_porcionado(in).csv").dropna(subset='pmra') # full gaia sample for one cluster
df = df.dropna(subset=['ra', 'dec', 'pmra','pmdec','parallax','bp_rp','phot_g_mean_mag'])
df.info(), df.columns

<class 'pandas.DataFrame'>
Index: 342356 entries, 0 to 365869
Data columns (total 29 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Unnamed: 0.1               342356 non-null  int64  
 1   Unnamed: 0                 342356 non-null  int64  
 2   source_id                  342356 non-null  int64  
 3   ra                         342356 non-null  float64
 4   dec                        342356 non-null  float64
 5   parallax                   342356 non-null  float64
 6   pmra                       342356 non-null  float64
 7   pmdec                      342356 non-null  float64
 8   ruwe                       342356 non-null  float64
 9   phot_g_mean_mag            342356 non-null  float64
 10  bp_rp                      342356 non-null  float64
 11  radial_velocity            164635 non-null  float64
 12  ra_error                   342356 non-null  float64
 13  dec_error                  342356 non-null  f

(None,
 Index(['Unnamed: 0.1', 'Unnamed: 0', 'source_id', 'ra', 'dec', 'parallax',
        'pmra', 'pmdec', 'ruwe', 'phot_g_mean_mag', 'bp_rp', 'radial_velocity',
        'ra_error', 'dec_error', 'pmra_error', 'pmdec_error', 'parallax_error',
        'visibility_periods_used', 'phot_bp_mean_mag', 'phot_rp_mean_mag',
        'nu_eff_used_in_astrometry', 'pseudocolour', 'ecl_lat',
        'astrometric_params_solved', 'zp', 'fidelity_v2', 'parallax_',
        'parallax_over_error', 'cell_id'],
       dtype='str'))

In [28]:

# features derivadas
df["dist_pc"] = 1000 / df["parallax"]
df["parallax_over_error"] = df["parallax_error"] / df["parallax"]

# filtro global (opcional pero recomendable)
df = df[df["parallax_over_error"] <= 0.1]

print("Total estrellas:", len(df))

Total estrellas: 342356


In [29]:
clustering_on = ['pmra','pmdec','dist_pc']

results = []

In [30]:
for cell_id in tqdm(sorted(df["cell_id"].unique())):

    df_cell = df[df["cell_id"] == cell_id].copy()

    n_stars = len(df_cell)

    print("\n==============================")
    print(f"Cell {cell_id}")
    print(f"N estrellas: {n_stars}")

    if n_stars < 30:
        print("Muy pocas estrellas, se omite")
        continue

    # ------------------
    # preparar datos
    # ------------------

    features = df_cell[clustering_on]
    data_scaled = RobustScaler().fit_transform(features)

    # ------------------
    # HDBSCAN
    # ------------------

    hd = HDBSCAN(
        min_cluster_size=45,
        min_samples=5,
        metric='euclidean'
    ).fit(data_scaled)

    labels = hd.labels_
    df_cell["label_hb"] = labels

    # ------------------
    # número de clusters
    # ------------------

    unique_labels = set(labels)
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)

    print(f"N clusters encontrados: {n_clusters}")

    # ------------------
    # silhouette
    # ------------------

    if n_clusters > 1:
        sil = silhouette_samples(data_scaled, labels)
        df_cell["silhouette"] = sil
    else:
        df_cell["silhouette"] = np.nan

    # ------------------
    # centrar VPD
    # ------------------

    pmra_center = df_cell["pmra"].median()
    pmdec_center = df_cell["pmdec"].median()

    df_cell["pmra_rel"] = df_cell["pmra"] - pmra_center
    df_cell["pmdec_rel"] = df_cell["pmdec"] - pmdec_center

    # ------------------
    # mapa de colores clusters
    # ------------------

    unique_labels = np.unique(labels)
    cmap = plt.cm.get_cmap('tab10', len(unique_labels))

    color_map = {}
    for i, lab in enumerate(unique_labels):
        if lab == -1:
            color_map[lab] = 'lightgray'
        else:
            color_map[lab] = cmap(i)

    # ======================
    # FIGURA 4 PANELES
    # ======================

    fig, axes = plt.subplots(2,2, figsize=(12,10))

    # ===== CMD por cluster =====
    ax = axes[0,0]

    for lab in unique_labels:
        cond = df_cell["label_hb"] == lab
        ax.scatter(
            df_cell["bp_rp"][cond],
            df_cell["phot_g_mean_mag"][cond],
            s=5,
            color=color_map[lab],
            label=f"{lab} ({cond.sum()})"
        )

    ax.invert_yaxis()
    ax.set_xlabel("BP-RP")
    ax.set_ylabel("G")
    ax.set_title("CMD por cluster")
    ax.legend(markerscale=3, fontsize=6)

    # ===== CMD con silueta =====
    ax = axes[0,1]

    sc1 = ax.scatter(
        df_cell["bp_rp"],
        df_cell["phot_g_mean_mag"],
        c=df_cell["silhouette"],
        cmap="viridis",
        s=5
    )

    ax.invert_yaxis()
    ax.set_xlabel("BP-RP")
    ax.set_ylabel("G")
    ax.set_title("CMD (silueta)")
    fig.colorbar(sc1, ax=ax)

    # ===== VPD por cluster =====
    ax = axes[1,0]

    for lab in unique_labels:
        cond = df_cell["label_hb"] == lab
        ax.scatter(
            df_cell["pmra_rel"][cond],
            df_cell["pmdec_rel"][cond],
            s=5,
            color=color_map[lab],
            label=f"{lab}"
        )

    ax.set_xlabel("pmRA (centrado)")
    ax.set_ylabel("pmDEC (centrado)")
    ax.set_title("VPD por cluster")
    ax.legend(markerscale=3, fontsize=6)

    # ===== VPD con silueta =====
    ax = axes[1,1]

    sc2 = ax.scatter(
        df_cell["pmra_rel"],
        df_cell["pmdec_rel"],
        c=df_cell["silhouette"],
        cmap="viridis",
        s=5
    )

    ax.set_xlabel("pmRA (centrado)")
    ax.set_ylabel("pmDEC (centrado)")
    ax.set_title("VPD (silueta)")
    fig.colorbar(sc2, ax=ax)

    plt.suptitle(f"Cell {cell_id} | N={n_stars} | clusters={n_clusters}")

    plt.tight_layout()
    plt.close()

    results.append(df_cell)


# ======================
# DATASET FINAL
# ======================

df_final = pd.concat(results)

df_final.to_csv("clusters_64regiones_full.csv", index=False)

print("\nProceso terminado.")

  0%|          | 0/64 [00:00<?, ?it/s]


Cell 1
N estrellas: 1461
N clusters encontrados: 2


  2%|▏         | 1/64 [00:00<00:15,  4.19it/s]


Cell 2
N estrellas: 3967
N clusters encontrados: 2


  3%|▎         | 2/64 [00:00<00:21,  2.86it/s]

...


Cell 63
N estrellas: 4381
N clusters encontrados: 2


 98%|█████████▊| 63/64 [00:52<00:00,  1.39it/s]


Cell 64
N estrellas: 1520
N clusters encontrados: 2


100%|██████████| 64/64 [00:53<00:00,  1.21it/s]



Proceso terminado.
